In [1]:
from typing import Any, NamedTuple, Sequence

import jax
import jax.numpy as jnp
from brax import envs
from brax.envs.wrappers.training import AutoResetWrapper, EpisodeWrapper
from gymnax.environments import environment, spaces


class Transition(NamedTuple):
    mask: jnp.ndarray
    action: jnp.ndarray
    pre_action: jnp.ndarray
    reward: jnp.ndarray
    log_prob: jnp.ndarray
    obs: jnp.ndarray
    

class GymnaxWrapper(object):
    """Base class for Gymnax wrappers."""

    def __init__(self, env):
        self._env = env

    # provide proxy access to regular attributes of wrapped object
    def __getattr__(self, name):
        return getattr(self._env, name)
    

class VecEnv(GymnaxWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.reset = jax.vmap(self._env.reset, in_axes=(0))
        self.step = jax.vmap(self._env.step, in_axes=(0, 0, 0))



class ClipAction(GymnaxWrapper):
    def __init__(self, env, low=-1.0, high=1.0):
        super().__init__(env)
        self.low = low
        self.high = high

    def step(self, key, state, action):
        """TODO: In theory the below line should be the way to do this."""
        # action = jnp.clip(action, self.env.action_space.low, self.env.action_space.high)
        action = jnp.clip(action, self.low, self.high)
        return self._env.step(key, state, action)

class BraxGymnaxWrapper:
    def __init__(self, env_name, backend="positional"):
        env = envs.get_environment(env_name=env_name, backend=backend)
        env = EpisodeWrapper(env, episode_length=1000, action_repeat=1)
        env = AutoResetWrapper(env)
        self._env = env
        self.action_size = env.action_size
        self.observation_size = (env.observation_size,)

    def reset(self, key):
        state = self._env.reset(key)
        return state.obs, state

    def step(self, key, state, action):
        next_state = self._env.step(state, action)
        return next_state.obs, next_state, next_state.reward, next_state.done > 0.5,True, {} ### ADD PLACHOLDER FOR TRUNCATED

    @property
    def observation_space(self):
        return spaces.Box(
            low=-jnp.inf,
            high=jnp.inf,
            shape=(self._env.observation_size,),
        )
    @property
    def action_space(self):
        return spaces.Box(
            low=-1.0,
            high=1.0,
            shape=(self._env.action_size,),
        )

In [2]:
import argparse
import logging
import os
from collections import deque
from functools import partial

import gymnasium as gym
import jax
import jax.numpy as jnp
import numpy as np
import tqdm
import wandb
from jax import config

from jaxrl_m.dataset import ActorReplayBuffer, ReplayBuffer
from jaxrl_m.evaluation import EpisodeMonitor
from jaxrl_m.onsac_clean import *
from jaxrl_m.rollout import rollout_policy, rollout_policy2
from jaxrl_m.utils import str2bool
from jaxrl_m.wandb import default_wandb_config, get_flag_dict, setup_wandb

logging.basicConfig(level=logging.CRITICAL)
jax.config.update('jax_default_matmul_precision', 'float32')

# Set env variables
os.environ["WANDB_API_KEY"]="28996bd59f1ba2c5a8c3f2cc23d8673c327ae230"
os.environ['PYTHONHASHSEED'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

##############################
parser = argparse.ArgumentParser()

parser.add_argument('--seed',type=int,default=42) 

parser.add_argument('--algo_name', type=str, default='superppo', help='the name of the RL algorithm')
parser.add_argument('--project_name',type=str,default="single_exp") 

parser.add_argument('--env_name',type=str,default="Walker2d-v5") 
parser.add_argument('--max_steps',type=int,default=1_000_000) 
parser.add_argument('--max_episode_steps',type=int,default=1000) 
parser.add_argument('--num_rollouts',type=int,default=5) 
parser.add_argument('--gamma',type=float,default=0.995)
parser.add_argument('--healthy_reward',type=float,default=0.5) 
parser.add_argument('--entropy_coeff',type=float,default=0.5) 

parser.add_argument('--num_critics',type=int,default=2)
parser.add_argument('--discount_actor',type=str2bool,default=True)
parser.add_argument('--discount_entropy',type=str2bool,default=True) 
parser.add_argument('--on_policy_data',type=str2bool,default=False)
parser.add_argument('--adaptive_critics',type=str2bool,default=False) 
parser.add_argument('--min_target',type=str2bool,default=False)

parser.add_argument('--critic_lr',type=float,default=3e-4) 
parser.add_argument('--actor_lr',type=float,default=3e-4) 
parser.add_argument('--temperature',type=float,default=0.05)
parser.add_argument('--use_layer_norm',type=str2bool,default=True)

parser.add_argument('--momentum',type=float,default=0.) 
parser.add_argument('--num_actor_updates',type=int,default=5) 
parser.add_argument('--clipping_ratio',type=float,default=0.1) 
parser.add_argument('--hidden_dims',type=int,default=256) 
parser.add_argument('--episode_based',type=str2bool,default=True) 

args = parser.parse_args([])
print(args)

2024-11-14 13:19:53.839884: W external/xla/xla/service/gpu/nvptx_compiler.cc:836] The NVIDIA driver's CUDA version is 12.5 which is older than the PTX compiler version (12.6.68). Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


Namespace(seed=42, algo_name='superppo', project_name='single_exp', env_name='Walker2d-v5', max_steps=1000000, max_episode_steps=1000, num_rollouts=5, gamma=0.995, healthy_reward=0.5, entropy_coeff=0.5, num_critics=2, discount_actor=True, discount_entropy=True, on_policy_data=False, adaptive_critics=False, min_target=False, critic_lr=0.0003, actor_lr=0.0003, temperature=0.05, use_layer_norm=True, momentum=0.0, num_actor_updates=5, clipping_ratio=0.1, hidden_dims=256, episode_based=True)


In [3]:
import flashbax as fbx
from copy import deepcopy
import brax

rng = jax.random.PRNGKey(42)

##### Define environment

config= {"env_name":"Walker2d","num_envs":256,"num_steps":32}
env = BraxGymnaxWrapper("walker2d")
env = ClipAction(env)
env = VecEnv(env)

single_env = BraxGymnaxWrapper("walker2d")


# First define hyper-parameters of the buffer.
max_length =  10*config["num_envs"]*config["num_steps"] # Maximum length of buffer (max number of experiences stored within the state).
min_length = 5000 # Minimum number of experiences saved in the buffer state before we can sample.
sample_batch_size = 256 # Batch size of experience data sampled from the buffer.

add_sequences = False # Will we be adding data in sequences to the buffer?
add_batch_size = config["num_envs"] # Will we be adding data in batches to the buffer? 
                      # It is possible to add data in both sequences and batches. 
                      # If adding data in batches, what is the batch size that is being added each time?

# Instantiate the flat buffer, which is a Dataclass of pure functions.
buffer = fbx.make_item_buffer(max_length, min_length, sample_batch_size, add_sequences, add_batch_size)

example_transition = dict(
    observations=env.observation_space.sample(rng),
    actions=env.action_space.sample(rng),
    rewards=0.0,
    masks=1.0,
    next_observations=env.observation_space.sample(rng),
    pre_actions = env.action_space.sample(rng),
    discounts=1.0,
    log_probs=0.,
)


buffer_state = buffer.init(example_transition)


agent = create_learner(args.seed,
                    
                observations=example_transition['observations'][None],
                actions =example_transition['actions'][None],
                max_steps=1e6,
                discount=args.gamma,
                discount_actor=args.discount_actor,
                min_target=args.min_target,
                discount_entropy=args.discount_entropy,
                adaptive_critics=args.adaptive_critics,
                num_critics= args.num_critics,
                entropy_coeff=args.entropy_coeff,
                temperature=args.temperature,
                actor_lr=args.actor_lr,
                critic_lr=args.critic_lr,
                momentum=args.momentum,
                clipping_ratio=args.clipping_ratio,
                num_actor_updates=args.num_actor_updates,
                actor_hidden_dims=(args.hidden_dims,args.hidden_dims),
                critic_hidden_dims=(args.hidden_dims,args.hidden_dims),
                use_layer_norm= args.use_layer_norm,
                #**FLAGS.config
                )

Extra kwargs: {'max_steps': 1000000.0}


In [4]:
env.observation_size

(17,)

In [ ]:
# Instantiate the flat buffer, which is a Dataclass of pure functions.
max_length = 50000
add_batch_size = 5000
buffer = fbx.make_item_buffer(max_length, min_length, sample_batch_size, add_sequences, add_batch_size)

example_transition = dict(
    obs=jnp.zeros((env.observation_size[0]),dtype=float),
    action=jnp.zeros((env.action_size), dtype=float),
    reward=jnp.zeros((), dtype=float),
    done=jnp.zeros((), dtype=bool),
    next_obs=jnp.zeros((env.observation_size[0]),dtype=float),
    discount = jnp.zeros((), dtype=float),
)


buffer_state = buffer.init(example_transition)


batch = dict(
    obs=jnp.zeros((add_batch_size,env.observation_size[0]),dtype=float),
    action=jnp.zeros((add_batch_size,env.action_size), dtype=float),
    reward=jnp.zeros((add_batch_size), dtype=float),
    done=jnp.zeros((add_batch_size), dtype=bool),
    next_obs=jnp.zeros((add_batch_size,env.observation_size[0]),dtype=float),
    discount = jnp.zeros((add_batch_size), dtype=float),
)

buffer_state = buffer.init(example_transition)

for i in range(20):
    buffer_state = buffer.add(buffer_state,batch)
    print(buffer_state.index)

AssertionError: [Chex] Assertion assert_trees_all_equal_dtypes failed:  Trees 0 and 1 differ in leaves 'done': types: bool != float32.

In [ ]:
rng  = jax.random.PRNGKey(0)
rng,_rng = jax.random.split(rng)
rngs = jax.random.split(rng,config["num_envs"])
obsv, env_state = env.reset(rngs)
runner_state = (agent,env_state,buffer_state,obsv, rng)

def rollout_policy(runner_state,discount=0.995):

    def _env_step(runner_state, unused):
            
            agent,env_state,buffer_state, observations, rng = runner_state

            # SELECT ACTION
            rng, _rng = jax.random.split(rng)
            actions,log_probs,pre_actions = agent.sample_actions(observations,seed=rng) ##TODO: Check using one seed

            # STEP ENV
            rng_step = jax.random.split(_rng, config["num_envs"])
            next_observations, env_state, rewards, dones,_, info = env.step(
                rng_step, env_state, actions
            )
            masks = 1.0*(1-dones)
            
            transition = {
                "masks":masks, "actions":actions, "pre_actions":pre_actions, 
                "rewards":rewards, "log_probs":log_probs, "observations":observations,"next_observations":next_observations,"discounts":masks,
            }
         
            buffer_state = buffer.add(buffer_state,transition)
            runner_state = (agent,env_state,buffer_state, next_observations, rng)
            
            return runner_state, transition
        

    runner_state,transition = jax.lax.scan(_env_step, runner_state, None, config["num_steps"])

    return runner_state

rslt = rollout_policy(runner_state)

In [ ]:
import chex



class EvaluationOutput(NamedTuple):
    """Evaluation output."""

    learner_state: Any
    episode_metrics: Dict[str, chex.Array]

class EvalState(NamedTuple):
    """State of the evaluator."""

    key: chex.PRNGKey
    env_state: Any
    done: Any
    observation : chex.Array
    step_count: chex.Array
    episode_return: chex.Array

def get_ff_evaluator_fn(
    env,
    agent,
    config = {},
    log_solve_rate: bool = False,
    eval_multiplier: int = 1,
) :
    """Get the evaluator function for feedforward networks.

    Args:
        env (Environment): An environment instance for evaluation.
        act_fn (callable): The act_fn that returns the action taken by the agent.
        config (dict): Experiment configuration.
        eval_multiplier (int): A scalar that will increase the number of evaluation
            episodes by a fixed factor. The reason for the increase is to enable the
            computation of the `absolute metric` which is a metric computed and the end
            of training by rolling out the policy which obtained the greatest evaluation
            performance during training for 10 times more episodes than were used at a
            single evaluation step.
    """

    def eval_one_episode(   : EvalState) -> Dict:
        """Evaluate one episode. It is vectorized over the number of evaluation episodes."""

        def _env_step(eval_state: EvalState) -> EvalState:
            """Step the environment."""
            # PRNG keys.
            
            key, env_state, done,observation, step_count, episode_return = eval_state
            # Select action.
            key, policy_key = jax.random.split(key)

            ###############################
            action = agent.deterministic_action(observation)
            ###############################

            # Step environment.
           
            next_observation, env_state, rewards, done,_, info = env.step(key,env_state, action.squeeze())
     

            # Log episode metrics.
            episode_return += rewards
            step_count += 1
            eval_state = EvalState(key, env_state,1.*done,next_observation,step_count, episode_return)
            return eval_state

        def not_done(carry: Tuple) -> bool:
            """Check if the episode is done."""
            is_not_done: bool = (carry.done==0).squeeze()
            return is_not_done

       
        final_state = jax.lax.while_loop(not_done, _env_step, init_eval_state)


        eval_metrics = {
            "episode_return": final_state.episode_return,
            "episode_length": final_state.step_count,
        }
        # Log solve episode if solve rate is required.
        if log_solve_rate:
            eval_metrics["solve_episode"] = jnp.all(
                final_state.episode_return >= config.env.solved_return_threshold
            ).astype(int)

        return eval_metrics

    def evaluator_fn(key: chex.PRNGKey) -> EvaluationOutput[EvalState]:
        """Evaluator function."""

        # Initialise environment states and timesteps.
        n_devices = len(jax.devices())

        eval_batch = (8 // n_devices) * 1

        key, *env_keys = jax.random.split(key, eval_batch + 1)
        env_keys = jax.random.split(key, eval_batch)
        observations,env_states = jax.vmap(env.reset)(jnp.stack(env_keys),)
        # Split keys for each core.
        key, *step_keys = jax.random.split(key, eval_batch + 1)
        # Add dimension to pmap over.
        step_keys = jnp.stack(step_keys).reshape(eval_batch, -1)

        eval_state = EvalState(
            key=step_keys,
            env_state=env_states,
            observation= observations,
            done = jnp.zeros((eval_batch,)),
            step_count=jnp.zeros((eval_batch, 1)),
            episode_return=jnp.zeros((eval_batch, 1)),
        )

        eval_metrics = jax.vmap(
            eval_one_episode,
            in_axes=0,
            #axis_name="eval_batch",
        )(eval_state)

        return EvaluationOutput(
            learner_state=eval_state,
            episode_metrics=eval_metrics,
        )

    return evaluator_fn


#act_fct = agent.deterministic_action
evaluator_fn = get_ff_evaluator_fn(single_env,agent)
evaluator_output = jax.jit(evaluator_fn)(rng)  

In [ ]:
evaluator_output = evaluator_fn(rng)  

In [ ]:
buffer_state.current_index

In [ ]:
# wandb_config = {
#     'project': args.project_name,
#     'name':None,
#     'hyperparam_dict':args.__dict__,
#     }
# wandb_run = setup_wandb(**wandb_config)

max_steps,log_interval = args.max_steps,10000
unlogged_steps,i,n_grads = 0,0,0
num_steps = config["num_envs"]*config["num_steps"]

runner_state = (agent,env_state,buffer_state,obsv, _rng)

evaluator_fn = get_ff_evaluator_fn(agent,rng)

with tqdm.tqdm(total=max_steps) as pbar:
        
        while (i < max_steps):
                
                logging.debug('policy rollout')
                if args.on_policy_data: replay_buffer = replay_buffer.reset()
                agent,env_state,buffer_state,obsv, rng = rollout_policy(runner_state)
                                                              
            
                
                unlogged_steps += num_steps
                i+=num_steps
                pbar.update(int(num_steps))
                
            
                ### Update critics ###:
                logging.debug('update critics')
                
                for i in range(1):
                       rng,_ = jax.random.split(rng)
                       batch = buffer.sample(buffer_state,rng)
                       agent = agent.update_critics(batch.experience)
                       
                evaluator_output = evaluator_fn(agent,rng)  
                episode_return = jnp.mean(evaluator_output.episode_metrics["episode_return"])
                print(f'episode_return {episode_return}')      
#                 ### Update actor ###
#                 actor_batch = actor_buffer.get_all()    
            
#                 agent, actor_update_info = agent.update_actor(actor_batch)    
                    
#                 critic_update_info = {}
#                 update_info = {**critic_update_info, **actor_update_info}
#                 n_grads += 1
                
#                 ### Log training info ###
#                 exploration_metrics = {f'exploration/disc_return': policy_return}
#                 train_metrics = {f'training/{k}': v for k, v in update_info.items()}
#                 train_metrics['training/undisc_return'] = undisc_policy_return
                
#                 wandb.log(train_metrics, step=int(i),commit=False)
#                 wandb.log(exploration_metrics, step=int(i),commit=False)
            
#                 ### Log evaluation info ###
                
#                 if unlogged_steps >= log_interval:
                    
#                     _,_,policy_return,undisc_policy_return = rollout_policy(
#                                                                     agent,eval_env,exploration_rng,
#                                                                     None,None,eval=True,
#                                                                     num_rollouts=10,
#                                                                     discount = args.gamma,max_length=1000)
#                     eval_metrics = {"policy_return": policy_return,"undisc_policy_return": undisc_policy_return}
                    
#                     eval_metrics = {f'evaluation/{k}': v for k, v in eval_metrics.items()}
#                     eval_metrics['n_grads']=int(n_grads)

#                     eval_step = i
#                     wandb.log(eval_metrics, step=int(eval_step),commit=True)
#                     unlogged_steps = 0
            
              
        
# wandb_run.finish()

In [ ]:
%time runner_state, traj_batch = jax.lax.scan(_env_step, runner_state, None, config["num_steps"])